In [ ]:
# Setup: find the course materials and load our helper code. Run this cell first.
import subprocess
import sys
from pathlib import Path

# Colab has a module called google.colab; a laptop does not.
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # On Colab: download the course materials once.
    TASK_DIR = Path("/content/intern-training")
    if not TASK_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "esm2-v1",
             "https://github.com/lagillenwater/intern-training", "/content/intern-training"],
            check=True,
        )
    # Install the exact package versions this course was tested with.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", 'accelerate==1.16.0', 'transformers==5.23.0', 'umap-learn==0.6.0'],
        check=True,
    )
else:
    # On a laptop: walk up from the current folder until we find the folder
    # whose code/ folder holds both helpers/ and checks.py.
    TASK_DIR = None
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "code" / "helpers").is_dir() and (folder / "code" / "checks.py").exists():
            TASK_DIR = folder
            break
    if TASK_DIR is None:
        raise RuntimeError("Could not find the course materials (a folder with code/helpers/ and code/checks.py).")

# Make our helper code importable, then load it.
sys.path.insert(0, str(TASK_DIR / "code"))
from helpers import *
from checks import *

print(f"Setup complete. Materials: {TASK_DIR}")


# 03 Fine-tune ESM-2

**What this step is for:** train ESM-2 itself to predict the enzyme class, using the same human training proteins the baseline learned from. You choose two training settings, run the training, read the loss curve, save the trained model, and save its predictions for every test protein so notebook 04 can compare it with the baseline.

**Time:** about 1.25 hours, including the time you wait while the model trains.

**Reads:** `data/enzymes.tsv`, and `predictions_baseline.tsv` from notebook 02. **Writes:** `checkpoints/`, `model_finetuned/`, and `predictions_finetuned.tsv` in your run folder on Google Drive.

**Before you start:** use a GPU runtime (in Colab: Runtime > Change runtime type > T4 GPU). Training on a CPU is much slower; if Colab has no GPU for you, see "No GPU available" in the README. Then run the Setup cell at the top and work top to bottom. If a cell below a `TODO` shows an error, first make sure that `TODO` passed its check.

In [ ]:
# Configuration for this run (the same in every notebook).
RUN_NAME = "esm2_8M"                      # a label for this run: the small ESM-2 model, with 8 million parameters
MODEL_NAME = "facebook/esm2_t6_8M_UR50D"  # the model's name on Hugging Face, the website it is downloaded from
WORK_DIR = setup_workdir(RUN_NAME)        # reconnects Google Drive on Colab and returns the run folder
print(WORK_DIR)

In [ ]:
# The standard data tools (notebook 00, Part 4), and the macro-F1 score from notebook 02.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

In [ ]:
df = load_dataset()  # the cleaned table: one row per enzyme, all five species

# The same three groups of proteins as in notebook 02, now as smaller tables.
train_df = df[df["split"] == "train"]  # human training proteins: the model learns from these
val_df = df[df["split"] == "val"]      # human validation proteins: used to pick the best checkpoint
test_df = df[df["split"] == "test"]    # every test protein, all five species: used only for the final scores
print(f"train: {len(train_df):,}   validation: {len(val_df):,}   test: {len(test_df):,}")

## What fine-tuning changes
In notebook 02, ESM-2 was frozen. It turned each protein into an embedding, and only the logistic regression on top learned anything: a few thousand weights.

Fine-tuning trains ESM-2 itself:
1. We put a new **classifier layer** on top of ESM-2: a small neural network that turns ESM-2's output into 7 class scores, one per class. (It reads the average of ESM-2's outputs over the protein's amino acids, the same summary the baseline used. So the only difference from the baseline is that ESM-2's own weights now change during training.) Its weights start out random.
2. Training shows the model a **batch** of proteins (8 at a time here), compares its guesses with the true classes, and measures how wrong it was with a number called the **loss**. Then it nudges every weight a little in the direction that lowers the loss. This is the backpropagation and gradient descent from your neural networks presentation. One batch and one update make one **step**.
3. "Every weight" means all of ESM-2's 8 million parameters, plus the new layer. Because ESM-2's own weights change, the way it reads proteins changes too: it can learn to pay attention to the parts of a sequence that matter for the enzyme class.

As in notebook 02, mistakes on rare classes cost more, so the model cannot do well by ignoring classes 4 to 7. Every epoch, the training also scores the model on the validation proteins and saves a **checkpoint**, a copy of the model at that point. At the end it keeps the checkpoint with the best validation macro-F1.

## Two training settings
- **Epochs** (`NUM_EPOCHS`): one epoch is one full pass through all the training proteins. With 5 epochs, the model sees each training protein 5 times. Too few, and it has not finished learning; too many, and it starts to memorize the training proteins instead of learning patterns that carry over to new ones.
- **Learning rate** (`LEARNING_RATE`): how big a step each update takes. Too big, and training jumps around and can wreck what ESM-2 already knew about proteins; too small, and it learns so slowly that it gets nowhere in the epochs you give it. Fine-tuning a pretrained model usually uses a small value.

`5e-5` is scientific notation for 5 times 10 to the power -5, which is 0.00005. Java writes it the same way: `double lr = 5e-5;`. Names in ALL CAPS are Python's habit for settings that stay fixed, like Java's `static final` constants.

In [ ]:
# TODO: Choose the training settings: NUM_EPOCHS (how many passes through the training proteins, a whole number)
#       and LEARNING_RATE (how big each update step is).
# Hint: start with NUM_EPOCHS = 5 and LEARNING_RATE = 5e-5
NUM_EPOCHS = None  # replace None with your value
LEARNING_RATE = None  # replace None with your value

In [ ]:
# As in notebook 02, the semicolon hides the extra True or False.
check_hyperparameters(NUM_EPOCHS, LEARNING_RATE);

## The score to beat
This cell loads the baseline's predictions that notebook 02 saved and scores them again on the human test proteins.

In [ ]:
# require_file stops with a clear message if notebook 02 has not saved the file yet.
baseline_path = require_file(WORK_DIR / "predictions_baseline.tsv", made_by="Notebook 02 (baseline)")
baseline_preds = pd.read_csv(baseline_path, sep="\t")
baseline_human = baseline_preds[baseline_preds["species"] == "human"]  # the file holds only test proteins
baseline_f1 = f1_score(baseline_human["y_true"], baseline_human["y_pred_baseline"], average="macro")
print(f"Baseline macro-F1 on human test proteins: {baseline_f1:.3f}")

**Prediction (write this before you start the training):** Will the fine-tuned model score higher than the baseline on the human test proteins, lower, or about the same? By roughly how much? Why?

## Set up the training
The **Trainer** is a Hugging Face tool that runs the training loop for us: batches, loss, updates, validation scores, and checkpoints. Our helper `make_trainer` builds one.

In [ ]:
# make_trainer (from our helpers) loads ESM-2 with a new, untrained classifier layer for our 7 classes,
# turns each sequence into tokens, and sets up the training with your two settings.
# Checkpoints are saved in WORK_DIR / "checkpoints" on your Drive.
trainer = make_trainer(train_df, val_df, WORK_DIR / "checkpoints", MODEL_NAME,
                       num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE)
# np.ceil rounds up, because the last batch of an epoch can have fewer than 8 proteins.
print(f"{len(train_df):,} training proteins in batches of 8: {int(np.ceil(len(train_df) / 8))} steps per epoch")

## Train
Training takes a while, much longer than any cell so far. While it runs you see a progress bar with an estimate of the time left, and after each epoch a new row in a table: the training loss, the validation loss, and the validation macro-F1.

**Keep this Colab tab open while it trains.** Colab can disconnect a notebook whose tab is closed or left idle for a long time. If it disconnects during training, the training is lost: run the notebook again from the top. Once the model is saved (the `TODO` after the loss curve), it is safe on your Drive, and a disconnect after that loses nothing.

In [ ]:
# TODO: Start the training.
# Hint: call trainer.train()
# YOUR CODE HERE

## The loss curve
The **loss** measures how wrong the model's guesses are: it is small when the model gives the true class a high probability. The training records two kinds:
- **Training loss**, on the proteins the model is learning from. It is recorded every 10 steps and jumps up and down, because each value comes from only a few batches.
- **Validation loss**, on the validation proteins, which the model never learns from. It is recorded once per epoch.

How to read the two together:
- Both still falling steeply at the end: the model was still learning when training stopped (**under-fitting**). More epochs might help.
- Training loss keeps falling while validation loss levels off or rises: the model is memorizing the training proteins instead of learning patterns that carry over (**over-fitting**). More epochs would not help.

`loss_history_frame` collects the recorded values into a table. A row with training loss has no validation loss, and the other way around; pandas marks a missing value as `NaN` ("not a number").

In [ ]:
history = loss_history_frame(trainer)  # one row per recorded value: step, epoch, train_loss, eval_loss, eval_macro_f1

# dropna(subset=[...]) keeps only the rows where that column has a value.
train_rows = history.dropna(subset=["train_loss"])  # the training loss, every 10 steps
eval_rows = history.dropna(subset=["eval_loss"])    # the validation loss and macro-F1, once per epoch
eval_rows

A pandas table can draw itself: `table.plot(x="col1", y="col2")` draws `col2` against `col1` with matplotlib. It returns the plot's **axes** (the drawing area). To draw a second line on the same axes, pass them to the second `.plot` with `ax=ax`. `label=` names each line in the legend, and `marker="o"` draws a dot at each point, which helps for the validation loss, with only one point per epoch.

In [ ]:
# TODO: Plot training loss and validation loss against epoch on the same axes, then label both axes.
# Hint: train_rows.plot(x=..., y=..., label=...) returns ax; draw eval_rows on it with ax=ax and marker="o";
#       then plt.xlabel, plt.ylabel, plt.show().
# YOUR CODE HERE

## Save the model
At the end of training, the Trainer loaded the checkpoint with the best validation macro-F1, so `trainer` now holds the best version. Save it to your Drive: notebooks 04 and 05 use it, and once it is saved a Colab disconnect cannot lose it.

In [ ]:
# TODO: Save the trained model to your run folder, in a folder named model_finetuned.
# Hint: trainer.save_model(WORK_DIR / "model_finetuned")
# YOUR CODE HERE

In [ ]:
check_saved_model(WORK_DIR / "model_finetuned");

## Predictions for every test protein
Now make the same kind of table as `preds` in notebook 02, with the fine-tuned model's predictions. Our helper `predict_classes(model_path, sequences)` loads a saved model and returns its predicted class (1 to 7) for each sequence, in order. It takes a minute or two for all the test proteins.

If Colab disconnected after you saved the model, you do not need to train again: run the Setup cell and every cell above the heading "Set up the training" (they are quick), then continue from here.

In [ ]:
# TODO: Make a table named ft_preds with one row per test protein (every species) and four columns:
#       accession, species, y_true (the real class), and y_pred_finetuned (the fine-tuned model's predicted class).
# Hint: start from test_df[[...]].copy() as with preds in 02; y_true from test_df["ec_class"];
#       y_pred_finetuned from predict_classes(model folder, sequences).
ft_preds = None  # replace None with your code

In [ ]:
check_prediction_table(ft_preds, df, "finetuned", name="ft_preds");

In [ ]:
# Put the columns in the order notebook 04 expects, then save as a tab-separated file (as in notebook 02).
ft_preds = ft_preds[["accession", "species", "y_true", "y_pred_finetuned"]]
ft_preds.to_csv(WORK_DIR / "predictions_finetuned.tsv", sep="\t", index=False)
print(f"Saved {len(ft_preds):,} rows to {WORK_DIR / 'predictions_finetuned.tsv'}")
ft_preds.head()

In [ ]:
# Both models on the human test proteins.
ft_human = ft_preds[ft_preds["species"] == "human"]
finetuned_f1 = f1_score(ft_human["y_true"], ft_human["y_pred_finetuned"], average="macro")
print(f"Macro-F1 on human test proteins: baseline {baseline_f1:.3f}   fine-tuned {finetuned_f1:.3f}")

**Compare:** Was your prediction right? What surprised you? (Notebook 04 checks whether the difference is bigger than chance, and looks at the other species.)

## In your own words
**Question 1.** What changed in the model during fine-tuning that did not change in notebook 02? Point to the cell in this notebook where that change happens.

**Your answer:**

**Question 2.** Look at your loss curve. Does it show signs of over-fitting, under-fitting, or neither? Which parts of the curve tell you? What would you change if you trained again?

**Your answer:**

## Done
Your fine-tuned model and its predictions are saved on your Drive.

Next: notebook 04, where you score both models on all five species and test whether fine-tuning made a real difference.